# <실습예제>

In [28]:
import os
import requests
from dotenv import load_dotenv #.env 사용
from bs4 import BeautifulSoup # BeautifulSoup 사용
from openai import OpenAI # OpenAI 사용

#API키 env에서 불러오기
load_dotenv()

NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

#네이버 API 식 요청 방식(인증 관련 정보)
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'
URL = BASE_URL + NEWS_URL

headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

params = {
    'query': '엔화', # 원하는 주제
    'sort': 'sim', # 정렬 : 연관도순으로
    'dispaly' : 5, #기사 100개 모아서
}
#naver news api에서 response된 정보
res = requests.get(URL, headers=headers, params=params)


In [ ]:
# OpenAI 기능을 사용 할 클라이언트 생성
client = OpenAI(
  api_key = os.getenv('OPENAI_API_KEY')
)

In [ ]:
# 네이버인 뉴스 제목, 링크정보만 추출하는 함수
def extract_naver_news(res : requests.models.Response):
  data = res.json()['items']
  news_datas = []
  # response data에서 '네이버 기사'인 '제목', '링크'만 추출 
  for item in data:
    if 'naver' in item['link']:
      item['title'] = item['title'].replace('<b>','').replace('</b','').replace('&quot;', '')
      refined_item = {
        'title' : item['title'],
        'link' : item['link']
      }
      news_datas.append(refined_item)

  return news_datas

# 링크에서 네이버 기사 본문 내용만 추출하는 함수
def extract_news_content(url:str):
  # 네이버 뉴스가 아니면 예외처리로 에러 발생시키는 코드
  if 'n.news.naver.com' not in url:
    raise Exception('네이버 뉴스가 아닙니다')
  
  # url의 response 정보(html)
  res = requests.get(url)

  # res(html)을 해석가능하게 파싱
  soup = BeautifulSoup(res.text, 'html.parser')

  # 파싱한 html에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
  news_content = soup.select_one('#dic_area').text.strip()

  return news_content

# OpenAI GPT 이용해서 댓글 생성하는 함수
def create_comment(content:str):
  system_msg = '너는 매우 착한 댓글을 만들어주는 AI야. 기사내용을 보고 긍정적인 댓글을 생성해줘'
  user_msg = content

  #gpt한테 받은 response 정보
  gpt_res = client.responses.create(
    model = 'gpt-4.1-mini',
    # system message
    instructions = system_msg,
    # user message
    input = user_msg
  )
  
  return gpt_res.output_text


In [ ]:
# 뉴스 데이터 리스트에 제목, 링크정보 추가
news_datas = extract_naver_news(res)
# for news_data in news_datas: #-> 링크만 추출 테스트용
#  print(news_data['link'])
# 뉴스 데이터 리스트에 해당 링크의 본문 내용 정보 추가
for news_data in news_datas:
  news_data['content'] = extract_news_content(news_data['link'])#본문 내용추가
  news_data['comment'] = create_comment(news_data['content'])#댓글 생성후 추가

print (news_datas)


https://n.news.naver.com/mnews/article/003/0014151350?sid=104
https://n.news.naver.com/mnews/article/009/0005726477?sid=101
https://n.news.naver.com/mnews/article/018/0006358116?sid=101
https://n.news.naver.com/mnews/article/001/0016269394?sid=101
https://n.news.naver.com/mnews/article/015/0005324740?sid=104
https://n.news.naver.com/mnews/article/308/0000038674?sid=101
https://n.news.naver.com/mnews/article/421/0009129664?sid=101
https://n.news.naver.com/mnews/article/003/0014148511?sid=104
